<!-- nav -->
[← 5 · One framework, three languages](05_One_Framework_Three_Languages.ipynb) · [Home](../../README.md) · [7 · From one language to another →](07_From_One_Language_To_Another.ipynb)

# 6 · Trees on the wire

**The problem.** The analyzer is a Python program; the editor plugin that shows its findings is TypeScript. Both need
the same tree. Parsing twice risks two parsers disagreeing, and an ad hoc JSON format is one more thing to maintain.

Trees are mbse-schemas data: every kind has a meta-schema, so a tree saves and loads as JSON or YAML like any object
graph, and the Python and TypeScript implementations write it byte for byte the same.

In [ ]:
import hashlib

from mbse.Programs.Framework.Syntax import same
from mbse.Programs.Python import Python314, Syntax as P
from mbse.Schemas.Framework import JSON

module = Python314.parse("def area(w, h):\n    return w * h\n")

## Step 1: a snapshot

`JSON.ToJSON(P.LANGUAGE.Builders)` writes a tree reachable from its root. Each syntax node is an object tagged with its
`kind`, holding its attributes, and its children are entries naming the property they fill:

In [ ]:
text = JSON.ToJSON(P.LANGUAGE.Builders).Reachable(P.Module.Schema, module, indent=1)
print(text[:900], "...")

## Step 2: reading it back

`JSON.FromJSON` builds the tree again. `same` compares trees by structure, since syntax nodes are compared by identity:

In [ ]:
loaded = JSON.FromJSON(P.LANGUAGE.Builders).Reachable(P.Module.Schema, text)
print(loaded is module, same(loaded, module))
print(Python314.print(loaded))

## Step 3: the same bytes in both languages

The TypeScript implementation parses the same source to the same tree, and writes the same snapshot. Their fingerprints
show it: the TypeScript tutorial prints this same digest.

In [ ]:
bigger = Python314.parse('''import math


class Circle:
    def __init__(self, radius):
        self.radius = radius

    def area(self):
        return math.pi * self.radius ** 2


print(f"{Circle(2).area():.2f}")
''')
snapshot = JSON.ToJSON(P.LANGUAGE.Builders).Reachable(P.Module.Schema, bigger)
hashlib.sha256(snapshot.encode()).hexdigest()

## Step 4: the grammar is data too

A language describes itself: every kind, its category, its properties and where it exists. A tool in any language can
read it, to generate code or documentation:

In [ ]:
grammar = P.LANGUAGE.grammar()
binop = next(kind for kind in grammar["kinds"] if kind["kind"] == "BinOp")
binop

## Why it works this way

- **One data model.** Trees are mbse-schemas objects, so they get its JSON and YAML, validation and comparison for
  free, and so does every tool built on mbse-schemas.
- **Byte-identical, by test.** Both implementations' snapshots of the shared corpus are compared byte for byte (the
  conformance suite), so a tree written by one reads in the other exactly.
- **Children as entries.** Each child records the property it fills and its index, so a snapshot keeps the tree's
  order and structure without a custom format.

<!-- nav -->
[← 5 · One framework, three languages](05_One_Framework_Three_Languages.ipynb) · [Home](../../README.md) · [7 · From one language to another →](07_From_One_Language_To_Another.ipynb)